# Helmet Compliance Detector — Colab Training Notebook

Trains both the YOLOv8 baseline and the YOLOv8+CBAM variant on a free-tier Colab GPU (T4).

Steps: mount Drive (optional, for persisting `runs/` and dataset across sessions) -> install deps -> set Kaggle credentials -> download/convert dataset -> train baseline -> train CBAM -> compare.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/helmet-detection'
import os
os.makedirs(PROJECT_DIR, exist_ok=True)

## Get the project code

Either upload this whole repo folder to `PROJECT_DIR` via the Drive UI/`Files` pane once, or (if you've pushed it to GitHub) clone it. Then `%cd` into it.

In [ ]:
%cd {PROJECT_DIR}
# If cloning from your own GitHub fork instead of uploading manually:
# !git clone <your-repo-url> .
!pip install -q -r requirements.txt

## Kaggle credentials

Upload your `kaggle.json` (from https://www.kaggle.com/settings -> API -> Create New Token) when prompted.

In [ ]:
from google.colab import files
import os
os.makedirs('/root/.kaggle', exist_ok=True)
uploaded = files.upload()  # select kaggle.json
for fname in uploaded:
    os.rename(fname, '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

In [ ]:
!python data/prepare_dataset.py --out data/dataset --val-frac 0.1 --test-frac 0.1

## Train baseline YOLOv8s

In [ ]:
!python train.py --variant baseline --data data/dataset/data.yaml --model-size s --epochs 60 --imgsz 640 --batch 16

## Train YOLOv8s + CBAM (novel-method variant)

In [ ]:
!python train.py --variant cbam --data data/dataset/data.yaml --model-size s --epochs 60 --imgsz 640 --batch 16

## Compare baseline vs CBAM on the held-out test split

In [ ]:
!python evaluate.py \
  --weights runs/detect/helmet-baseline-yolov8s/weights/best.pt runs/detect/helmet-cbam-yolov8s/weights/best.pt \
  --names baseline cbam \
  --data data/dataset/data.yaml \
  --out docs/results_comparison.csv

## (Optional) Quick sanity check on a sample image

In [ ]:
from ultralytics import YOLO
from models import register_modules  # only needed for the CBAM checkpoint
model = YOLO('runs/detect/helmet-cbam-yolov8s/weights/best.pt')
results = model.predict('data/dataset/test/images', save=True, conf=0.35)
print('Annotated predictions saved under runs/detect/predict*/')